In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras import layers, models, optimizers, callbacks
from sklearn.metrics import f1_score

# Adjust these to your actual paths
train_dir = '/content/drive/MyDrive/X-ray_chest/Dataset/Training_Dataset_TB'
test_dir = '/content/drive/MyDrive/X-ray_chest/Dataset/Test_Dataset_TB'
csv_path = '/content/drive/MyDrive/X-ray_chest/Dataset/TB_train.csv'

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_TOP = 10
EPOCHS_FINE = 10
SEED = 42

In [ ]:
train_df = pd.read_csv(csv_path)
print("Original columns:", train_df.columns)

# Rename CSV columns to match x_col and y_col in flow_from_dataframe
train_df.rename(columns={'ID': 'image', 'Target': 'class'}, inplace=True)

# Convert the label to string (flow_from_dataframe often expects a string for class_mode='binary')
train_df['class'] = train_df['class'].astype(str)

print("Renamed columns:", train_df.columns)
print("DataFrame head:\n", train_df.head())


Original columns: Index(['ID', 'Target'], dtype='object')
Renamed columns: Index(['image', 'class'], dtype='object')
DataFrame head:
                image class
0  TB_train_0001.png     1
1  TB_train_0002.png     0
2  TB_train_0003.png     0
3  TB_train_0004.png     0
4  TB_train_0005.png     0


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_dir,     # The folder containing the training images
    x_col='image',           # The column with image filenames
    y_col='class',           # The column with class labels (now "class")
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    seed=SEED
)

validation_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_dir,
    x_col='image',
    y_col='class',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False,
    seed=SEED
)


Found 2352 validated image filenames belonging to 2 classes.
Found 588 validated image filenames belonging to 2 classes.


In [ ]:
# Example: Build an EfficientNetB3 model
base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = models.Model(inputs, outputs)

# Compile
model.compile(
    optimizer=optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train (top layers)
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS_TOP
)

# Fine-tune (unfreeze some layers, etc.):
base_model.trainable = True
for layer in base_model.layers[:-100]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS_TOP + EPOCHS_FINE
)


43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
47/74 ━━━━━━━━━━━━━━━━━━━━ 8:18 18s/step - accuracy: 0.7446 - loss: 0.5246

KeyError: 'class'